# DAICO Navigator - an Agentic AI Assistant for the DAICO Center

**Course:** Building Agentic AI Systems (AGT-401) - **Capstone Project**

**Programme:** SDAIA Academy - *Building Agentic AI Systems*, hosted at **DAICO** (Data & AI Center of Excellence), **King Saud University (KSU)**, sponsored by **SDAIA**
**Cohort:** 16-20 August 2026 (5 days)
**Declared track:** **Track A - Supervisor** (with a **Track B** human handoff for enrollment)

**Team:**
- Hamed Ahmed Aldkhyyal
- Saif Fawaz Alanzie
- Fahad Abdullah Alanazi
- Yousef Farhan Alanzie

---

**What it is.** DAICO Navigator is a multi-agent assistant for the DAICO training center. A supervisor LLM
routes each learner message to one of four workers - **Course Advisor** (real catalog tools), **FAQ**
(RAG over center documents), **Capstone Mentor** (RAG over the rubric), and **Enrollment** (human-in-the-loop
registration). It remembers each learner across conversations and is built on the LangGraph Functional API.

## How this notebook maps to the 8 rubric sections

| # | Rubric section | Where in this notebook |
|---|---|---|
| 1 | Agent fundamentals (real tools + structured output) | §1 |
| 2 | Multi-agent / routing (LLM supervisor, Track A) | §2 |
| 3 | RAG pipeline (load→split→embed→store→retrieve) | §3 |
| 4 | Context & state (checkpointer + Store, cross-thread test) | §4 |
| 5 | Human-in-the-loop (interrupt + resume) | §5 |
| 6 | Functional API & error handling (@task/@entrypoint + 2 strategies) | §6 |
| 7 | Workflow pattern (Evaluator-Optimizer, named) | §7 |
| 8 | LangSmith observability | §8 |

Sections are presented in **dependency order** (each builds on the last), but every header names its rubric number.

## Setup - environment, model, and observability switch

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"

# Enable LangSmith tracing when a key is available.
if os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ.setdefault("LANGCHAIN_PROJECT", "daico-navigator")
    print("LangSmith tracing: ON  ->  project:", os.environ["LANGCHAIN_PROJECT"])
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing: OFF")

DATA_DIR = Path("data") if Path("data").exists() else (Path("..") / "data")
print("Data directory:", DATA_DIR.resolve())

LangSmith tracing: ON  ->  project: daico-navigator
Data directory: C:\Users\hamad\OneDrive\Documents\GitHub\AGENTIC-AI-DAICO-PROJECT\data


In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# One OpenAI key powers both the chat model and the embeddings used by the RAG pipeline.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("LLM smoke test ->", llm.invoke("Reply with the single word: ready").content)

LLM smoke test -> ready


## Rubric §1 - Agent fundamentals: real tools + structured output

The tools below read the **real** DAICO catalog (`data/courses.json`) and use their arguments - no hardcoded
strings. The model *chooses* which to call. We also show `with_structured_output` returning a Pydantic object
that downstream **code** parses (not a human).

In [3]:
import re
from langchain_core.tools import tool

CATALOG = json.loads((DATA_DIR / "courses.json").read_text(encoding="utf-8"))
COURSES = {c["course_id"]: c for c in CATALOG["courses"]}

@tool
def search_courses(query: str) -> str:
    "Search the DAICO catalog by keywords in title, description, level, or track. Returns matching courses as JSON."
    words = [w for w in re.findall(r"[a-z0-9]+", query.lower()) if len(w) > 2]
    hits = []
    for c in CATALOG["courses"]:
        haystack = (c["title"] + " " + c["description"] + " " + c["level"] + " " + c["track"]).lower()
        if any(w in haystack for w in words):
            hits.append({"course_id": c["course_id"], "title": c["title"],
                         "level": c["level"], "track": c["track"]})
    return json.dumps(hits, ensure_ascii=False)

@tool
def get_course_details(course_id: str) -> str:
    "Return the full catalog record for a course id such as ML-201."
    c = COURSES.get(course_id.upper())
    return json.dumps(c, ensure_ascii=False) if c else "No course with id " + course_id

@tool
def check_seat_availability(course_id: str) -> str:
    "Return how many seats remain for a course id."
    c = COURSES.get(course_id.upper())
    if not c:
        return "No course with id " + course_id
    n = c["seats_available"]
    status = "FULL - waitlist only" if n == 0 else str(n) + " seat(s) available"
    return c["course_id"] + " (" + c["title"] + "): " + status

@tool
def get_prerequisites(course_id: str) -> str:
    "Return the prerequisite course ids (with titles) for a course id."
    c = COURSES.get(course_id.upper())
    if not c:
        return "No course with id " + course_id
    prereqs = [{"course_id": p, "title": COURSES.get(p, {}).get("title", "?")} for p in c["prerequisites"]]
    return json.dumps({"course_id": c["course_id"], "prerequisites": prereqs}, ensure_ascii=False)

TOOLS = [search_courses, get_course_details, check_seat_availability, get_prerequisites]
print("Registered tools:", [t.name for t in TOOLS])

Registered tools: ['search_courses', 'get_course_details', 'check_seat_availability', 'get_prerequisites']


In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm_with_tools = llm.bind_tools(TOOLS)
TOOL_MAP = {t.name: t for t in TOOLS}

def run_agent(question: str, verbose: bool = True) -> str:
    "Minimal tool-calling loop: the LLM decides which tools to call and when to stop."
    messages = [
        SystemMessage(content="You are the DAICO course assistant. Use the tools to answer with real catalog "
                              "data. Never invent course facts. Be concise."),
        HumanMessage(content=question),
    ]
    for _ in range(5):
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.content
        for tc in ai.tool_calls:
            if verbose:
                print("  -> tool call:", tc["name"], tc["args"])
            result = TOOL_MAP[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return messages[-1].content

answer = run_agent("How many seats are left in Deep Learning, and what must I take before I can enroll?")
print("\nANSWER:\n", answer)

  -> tool call: search_courses {'query': 'Deep Learning'}
  -> tool call: search_courses {'query': 'Deep Learning'}


  -> tool call: check_seat_availability {'course_id': 'DL-301'}
  -> tool call: get_prerequisites {'course_id': 'DL-301'}



ANSWER:
 In the "Deep Learning with PyTorch" course (DL-301), there are 3 seats available. Before enrolling, you must complete the prerequisite course "Applied Machine Learning" (ML-201).


In [5]:
from typing import List, Literal
from pydantic import BaseModel, Field

class CourseRecommendation(BaseModel):
    "A structured recommendation consumed by code, not read by a human."
    course_id: str = Field(description="Recommended course id, e.g. ML-201")
    title: str = Field(description="The course title")
    reason: str = Field(description="One sentence on why it fits the learner")
    prerequisites: List[str] = Field(description="Prerequisite course ids to finish first")

recommender = llm.with_structured_output(CourseRecommendation)
catalog_brief = "\n".join(
    c["course_id"] + ": " + c["title"] + " (" + c["level"] + ", prereqs=" + str(c["prerequisites"]) + ")"
    for c in CATALOG["courses"]
)
rec = recommender.invoke(
    "Catalog:\n" + catalog_brief +
    "\n\nLearner: 'I know Python and basic ML and want to build LLM agents.' Recommend exactly one course."
)
print("Type:", type(rec).__name__)
print("Parsed by code ->", "course_id:", rec.course_id, "| prerequisites:", rec.prerequisites)
print("Reason:", rec.reason)

Type: CourseRecommendation
Parsed by code -> course_id: AGT-401 | prerequisites: ['ML-201']
Reason: This course focuses on building intelligent agents, which aligns perfectly with your goal of creating LLM agents.


## Rubric §3 - RAG pipeline

Real DAICO documents are **loaded → split → embedded → stored → retrieved**. We then prove retrieval works by
asking a question whose answer is *verbatim* in the documents.

In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

# 1) LOAD
doc_files = ["about_daico.md", "policies.md", "faq.md", "capstone_rubric.md"]
raw_docs = [Document(page_content=(DATA_DIR / f).read_text(encoding="utf-8"), metadata={"source": f})
            for f in doc_files]
print("Loaded", len(raw_docs), "documents:", [d.metadata["source"] for d in raw_docs])

# 2) SPLIT
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(raw_docs)
print("Split into", len(chunks), "chunks")

# 3) EMBED + 4) STORE
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)
print("Embedded and stored", len(chunks), "chunks")

# 5) RETRIEVE
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Retriever ready.")

Loaded 4 documents: ['about_daico.md', 'policies.md', 'faq.md', 'capstone_rubric.md']
Split into 19 chunks


Embedded and stored 19 chunks
Retriever ready.


In [7]:
# Rubric warning: ask a question whose answer is VERBATIM in the docs to prove the pipeline is not broken.
probe = "What attendance percentage is required to receive the certificate?"
hits = retriever.invoke(probe)
print("Top hit source:", hits[0].metadata["source"])
print("Snippet:", hits[0].page_content[:220].replace("\n", " "), "...")

Top hit source: policies.md
Snippet: ## Attendance - A minimum of **80% attendance** is required to be eligible for the completion certificate. - Sessions run for 5 days. Missing more than one full day makes a learner ineligible for the certificate for that ...


In [8]:
def rag_answer(question: str) -> str:
    "Retrieve relevant DAICO chunks and answer grounded strictly in them."
    docs = retriever.invoke(question)
    context = "\n\n".join("[" + d.metadata["source"] + "] " + d.page_content for d in docs)
    prompt = ("Answer using ONLY the context below. If the answer is not present, say you do not have that "
              "information.\n\nContext:\n" + context + "\n\nQuestion: " + question)
    return llm.invoke(prompt).content

print(rag_answer("What attendance percentage is required, and what score does a capstone need to pass?"))

A minimum of **80% attendance** is required, and a capstone needs to score **60 out of 100** to pass.


**RAG design choice - Hybrid RAG.** DAICO answers come from two very different stores: *structured* catalog
facts (seats, prerequisites - best served by the tools in §1) and *unstructured* prose (policies, FAQ, rubric -
best served by vector retrieval). A pure 2-Step RAG would miss live seat counts; pure Agentic RAG would waste
tool-calling turns on questions a single retrieval answers well. We therefore use **Hybrid RAG**: the §2
supervisor decides per message whether to use structured tools (Course Advisor) or vector retrieval (FAQ /
Capstone Mentor).

## Rubric §2 - Multi-agent routing (Track A: Supervisor)

The routing decision is made by the **LLM** via `with_structured_output` - not by keyword matching. A `Route`
Pydantic model constrains the destination to four workers.

In [9]:
class Route(BaseModel):
    "The supervisor's routing decision for a learner message."
    destination: Literal["course_advisor", "enrollment", "capstone_mentor", "faq"] = Field(
        description=("course_advisor for course discovery, recommendations, prerequisites, or seats; "
                     "enrollment for requests to register or enroll in a specific course; "
                     "capstone_mentor for questions about the capstone project or its rubric; "
                     "faq for general questions about DAICO, SDAIA, fees, location, or certificates")
    )
    reason: str = Field(description="One short sentence justifying the choice")

supervisor = llm.with_structured_output(Route)

tests = [
    "Which course should I take to learn how to build agents?",
    "How much does DAICO cost and where is it located?",
    "What does the capstone need in order to pass?",
    "I want to enroll in ML-201",
]
for t in tests:
    d = supervisor.invoke(t)
    print("[", d.destination.ljust(15), "]", t, "\n     reason:", d.reason)

[ course_advisor  ] Which course should I take to learn how to build agents? 
     reason: The learner needs guidance on course recommendations for building agents.


[ faq             ] How much does DAICO cost and where is it located? 
     reason: The question pertains to general information about DAICO's fees and location.


[ capstone_mentor ] What does the capstone need in order to pass? 
     reason: The question is specifically about the capstone project requirements.


[ enrollment      ] I want to enroll in ML-201 
     reason: The learner is requesting to register for a specific course.


## Rubric §4 - Context & state: checkpointer (short-term) + Store (long-term)

`InMemorySaver` gives each conversation a `thread_id` (short-term). A separate `InMemoryStore` holds durable
learner facts that must survive across conversations (long-term). The **cross-thread test** below writes a fact
in one thread and reads it back from a different thread.

In [10]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()   # short-term, per-thread conversation state
store = InMemoryStore()          # long-term, durable across threads

def remember(user_id: str, key: str, value) -> None:
    store.put(("learners", user_id), key, {"value": value})

def recall(user_id: str, key: str):
    item = store.get(("learners", user_id), key)
    return item.value["value"] if item else None

print("Checkpointer + Store initialised.")

Checkpointer + Store initialised.


In [11]:
@entrypoint(checkpointer=checkpointer, store=store)
def memory_demo(inputs: dict) -> dict:
    uid = inputs["user_id"]
    count = (recall(uid, "questions_asked") or 0) + 1
    remember(uid, "questions_asked", count)
    if "interest" in inputs:
        remember(uid, "interest", inputs["interest"])
    return {"user_id": uid, "questions_asked_total": count, "known_interest": recall(uid, "interest")}

cfg_a = {"configurable": {"thread_id": "conv-A"}}
cfg_b = {"configurable": {"thread_id": "conv-B"}}   # a DIFFERENT conversation

a = memory_demo.invoke({"user_id": "hamed", "interest": "agentic AI"}, cfg_a)
print("conv-A ->", a)
b = memory_demo.invoke({"user_id": "hamed"}, cfg_b)   # no interest passed here
print("conv-B ->", b)

assert b["known_interest"] == "agentic AI", "cross-thread fact was lost"
assert b["questions_asked_total"] == 2, "counter did not persist across threads"
print("\n[PASS] CROSS-THREAD MEMORY CONFIRMED: a fact written in conv-A was read back in conv-B.")

conv-A -> {'user_id': 'hamed', 'questions_asked_total': 1, 'known_interest': 'agentic AI'}
conv-B -> {'user_id': 'hamed', 'questions_asked_total': 2, 'known_interest': 'agentic AI'}

[PASS] CROSS-THREAD MEMORY CONFIRMED: a fact written in conv-A was read back in conv-B.


## Rubric §7 - Workflow pattern: **Evaluator-Optimizer**

We implement the **Evaluator-Optimizer** pattern for the Capstone Mentor. It fits because good capstone feedback
should be *drafted*, then *judged* against the rubric, then *revised* if it is not specific enough - a
generate → judge → refine loop. Built with the Functional API (`@task` / `@entrypoint`).

In [12]:
class Evaluation(BaseModel):
    "The evaluator's judgement of a draft piece of mentor feedback."
    verdict: Literal["accept", "revise"] = Field(
        description="accept if the feedback is specific, rubric-grounded and actionable; otherwise revise")
    critique: str = Field(description="What to improve, if revising")

evaluator = llm.with_structured_output(Evaluation)
RUBRIC_TEXT = (DATA_DIR / "capstone_rubric.md").read_text(encoding="utf-8")

@task
def draft_feedback(idea: str) -> str:
    return llm.invoke("You are a DAICO capstone mentor. Using this rubric:\n" + RUBRIC_TEXT[:2200] +
                      "\n\nGive a brief FIRST-PASS critique of the student's idea in about 3-4 sentences.\n\n"
                      "Idea: " + idea).content

@task
def judge_feedback(feedback: str) -> Evaluation:
    return evaluator.invoke(
        "You are a strict reviewer. Accept ONLY if the feedback gives concrete, actionable fixes for each of the "
        "8 rubric sections by name. A brief or high-level critique must be marked 'revise'.\n\nFeedback:\n" + feedback)

@task
def improve_feedback(feedback: str, critique: str) -> str:
    return llm.invoke("Improve this feedback. Address this critique: " + critique +
                      "\n\nOriginal feedback:\n" + feedback).content

@entrypoint()
def capstone_review_workflow(idea: str) -> str:
    fb = draft_feedback(idea).result()
    verdict = judge_feedback(fb).result()
    print("   [evaluator verdict:", verdict.verdict, "]", verdict.critique[:90])
    if verdict.verdict == "accept":
        return fb
    return improve_feedback(fb, verdict.critique).result()

flawed_idea = ("A chatbot that answers with if-else keyword rules and keeps a growing list of chat messages "
               "as its long-term memory.")
print(capstone_review_workflow.invoke(flawed_idea)[:600], "...")

   [evaluator verdict: revise ] The feedback lacks specific, actionable fixes for each of the 8 rubric sections. It mentio


To enhance the feedback and provide specific, actionable guidance for each of the eight rubric sections, I will break down the critique into clear categories and offer targeted suggestions for improvement.

---

**Feedback on Chatbot Project**

**1. Idea Development:**
The concept of a chatbot using if-else keyword rules is a good starting point, but it lacks the flexibility needed for effective interaction. To improve this section, consider exploring more advanced natural language processing techniques, such as machine learning models or libraries like Rasa or Dialogflow, which can help the c ...


## Rubric §6 - Functional API & error handling

Everything runs on the **Functional API** (`@task` / `@entrypoint`). We implement **two** of the four error
strategies:
- **Transient retry** - a real `RetryPolicy` object on a flaky task.
- **LLM-recoverable** - a structured-output task that feeds a `ValidationError` back to the model and retries.

In [13]:
from langgraph.types import RetryPolicy

_attempts = {"n": 0}

@task(retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.2))
def fetch_seat_snapshot(course_id: str) -> str:
    "Simulates a flaky network lookup; the RetryPolicy handles the transient failures."
    _attempts["n"] += 1
    if _attempts["n"] < 3:                       # fail on attempts 1 and 2, succeed on 3
        raise ConnectionError("transient network error (attempt " + str(_attempts["n"]) + ")")
    return check_seat_availability.invoke({"course_id": course_id})

@entrypoint(checkpointer=InMemorySaver())
def retry_demo(course_id: str) -> str:
    return fetch_seat_snapshot(course_id).result()

print(retry_demo.invoke("ML-201", {"configurable": {"thread_id": "retry-1"}}))
print("Error strategy 1 (transient retry): RetryPolicy recovered after", _attempts["n"], "attempts.")

ML-201 (Applied Machine Learning): 5 seat(s) available
Error strategy 1 (transient retry): RetryPolicy recovered after 3 attempts.


In [14]:
from pydantic import ValidationError

@task
def classify_message(text: str) -> dict:
    "Route with structured output; on a ValidationError, feed the error back and retry (LLM-recoverable)."
    feedback = ""
    for _ in range(3):
        try:
            r = supervisor.invoke("Classify this learner message." + feedback + "\n\nMessage: " + text)
            return {"destination": r.destination, "reason": r.reason}  # plain dict keeps checkpoints clean
        except ValidationError as e:
            feedback = "\n\nYour previous answer was invalid: " + str(e) + " Return a valid destination."
    return {"destination": "faq", "reason": "safe default after repeated invalid classifications"}

@entrypoint(checkpointer=InMemorySaver())
def classify_demo(text: str) -> dict:
    return classify_message(text).result()

print(classify_demo.invoke("I want to sign up for ML-201", {"configurable": {"thread_id": "cls-1"}}))
print("Error strategy 2 (LLM-recoverable): structured classifier with validation feedback loop.")

{'destination': 'enrollment', 'reason': 'The learner is requesting to register for a specific course.'}
Error strategy 2 (LLM-recoverable): structured classifier with validation feedback loop.


## Rubric §5 - Human-in-the-loop: enrollment (Track B handoff)

Enrollment is irreversible-ish (it reserves a sponsored seat), so the agent **pauses** with `interrupt()` and
waits for a human to approve before committing. The run is completed with `Command(resume=...)`.

In [15]:
from langgraph.types import interrupt, Command

class EnrollmentRequest(BaseModel):
    "Structured extraction of an enrollment request."
    course_id: str = Field(description="The course id to enroll in, e.g. ML-201")
    student_name: str = Field(description="The learner's full name if stated, else 'unknown'")

enroll_extractor = llm.with_structured_output(EnrollmentRequest)

@task
def bump_and_get_count(uid: str) -> int:
    "Increment the learner's long-term question counter (wrapped in a task so resume does not double-count)."
    count = (recall(uid, "questions_asked") or 0) + 1
    remember(uid, "questions_asked", count)
    return count

@task
def enroll_task(question: str, uid: str) -> str:
    req = enroll_extractor.invoke("Extract the enrollment request from: " + question)
    c = COURSES.get(req.course_id.upper())
    if not c:
        return "I could not find course " + req.course_id + ". Please provide a valid course id."
    if c["seats_available"] == 0:
        return c["course_id"] + " (" + c["title"] + ") is FULL; you would be placed on the waitlist."
    # HUMAN-IN-THE-LOOP: pause before the irreversible enrollment.
    decision = interrupt({
        "action": "confirm_enrollment",
        "message": "Confirm enrollment of " + req.student_name + " into " + c["course_id"] +
                   " (" + c["title"] + ")? Reply 'approve' or 'cancel'.",
        "course_id": c["course_id"],
        "seats_available": c["seats_available"],
    })
    if str(decision).strip().lower() == "approve":
        remember(uid, "enrolled_course", c["course_id"])
        return "Enrollment confirmed: " + req.student_name + " is registered in " + c["course_id"] + \
               " (" + c["title"] + "). Confirmation email within 24h."
    return "Enrollment cancelled for " + c["course_id"] + "."

print("Enrollment task defined.")

Enrollment task defined.


In [16]:
# ---- The integrated DAICO Navigator: supervisor + workers + memory, all on the Functional API ----
@task
def advisor_task(q: str) -> str:
    return run_agent(q, verbose=False)

@task
def faq_task(q: str) -> str:
    return rag_answer(q)

@entrypoint(checkpointer=checkpointer, store=store)
def daico_navigator(inputs: dict) -> dict:
    q, uid = inputs["question"], inputs["user_id"]
    count = bump_and_get_count(uid).result()          # long-term memory (survives across threads)
    route = classify_message(q).result()              # §2 routing + §6 error strategy 2
    dest = route["destination"]

    if dest == "course_advisor":
        answer = advisor_task(q).result()             # §1 real tools
    elif dest == "enrollment":
        answer = enroll_task(q, uid).result()         # §5 human-in-the-loop
    elif dest == "capstone_mentor":
        answer = faq_task(q).result()                 # §3 RAG over the rubric
    else:
        answer = faq_task(q).result()                 # §3 RAG over policies/FAQ

    return {"destination": dest, "reason": route["reason"], "answer": answer, "your_question_count": count}

print("daico_navigator entrypoint ready.")

daico_navigator entrypoint ready.


In [17]:
# --- Fire the interrupt: the run PAUSES awaiting human approval ---
enroll_cfg = {"configurable": {"thread_id": "enroll-demo-1"}}
paused = daico_navigator.invoke(
    {"user_id": "saif", "question": "I'd like to enroll in ML-201. My name is Saif Fawaz Alanzie."},
    enroll_cfg,
)
assert "__interrupt__" in paused, "expected an interrupt"
print("PAUSED - awaiting human approval:")
print(json.dumps(paused["__interrupt__"][0].value, indent=2, ensure_ascii=False))

PAUSED - awaiting human approval:
{
  "action": "confirm_enrollment",
  "message": "Confirm enrollment of Saif Fawaz Alanzie into ML-201 (Applied Machine Learning)? Reply 'approve' or 'cancel'.",
  "course_id": "ML-201",
  "seats_available": 5
}


In [18]:
# --- Resume with the human decision: the run COMPLETES ---
resumed = daico_navigator.invoke(Command(resume="approve"), enroll_cfg)
print("RESUMED RESULT:")
print(json.dumps(resumed, indent=2, ensure_ascii=False))
print("\nLong-term record - Saif's enrolled course:", recall("saif", "enrolled_course"))

RESUMED RESULT:
{
  "destination": "enrollment",
  "reason": "The learner is requesting to enroll in a specific course.",
  "answer": "Enrollment confirmed: Saif Fawaz Alanzie is registered in ML-201 (Applied Machine Learning). Confirmation email within 24h.",
  "your_question_count": 1
}

Long-term record - Saif's enrolled course: ML-201


## Rubric §8 - LangSmith observability

Tracing uses the `LANGCHAIN_TRACING_V2` variable and runs whenever a LangSmith key is configured. The cell
below issues a traced run and reports the project it was sent to.

In [19]:
if os.environ.get("LANGCHAIN_TRACING_V2") == "true":
    _ = daico_navigator.invoke(
        {"user_id": "trace", "question": "What is DAICO and is it free?"},
        {"configurable": {"thread_id": "trace-1"}},
    )
    print("Traced run complete. Project:", os.environ.get("LANGCHAIN_PROJECT"))
    print("The trace shows the supervisor classification, the chosen worker, and each LLM call.")
else:
    print("LangSmith tracing is OFF.")

Traced run complete. Project: daico-navigator
The trace shows the supervisor classification, the chosen worker, and each LLM call.


## End-to-end demo - routing + cross-thread memory on the real agent

In [20]:
demo_qs = [
    ("yousef", "Which course teaches building AI agents, and how many seats are left in it?"),
    ("yousef", "Is DAICO free, and where is it located?"),
    ("yousef", "What does the capstone need in order to pass?"),
]
for i, (uid, q) in enumerate(demo_qs):
    r = daico_navigator.invoke({"user_id": uid, "question": q},
                               {"configurable": {"thread_id": "demo-" + str(i)}})
    print("\nQ:", q)
    print("  routed to [" + r["destination"] + "] (question #" + str(r["your_question_count"]) + ")")
    print("  " + r["answer"][:280].replace("\n", " "))

print("\nCross-thread memory: Yousef's question count across 3 different threads =",
      recall("yousef", "questions_asked"))


Q: Which course teaches building AI agents, and how many seats are left in it?
  routed to [course_advisor] (question #1)
  The course that teaches building AI agents is **"Building Agentic AI Systems" (AGT-401)**. There are **6 seats available** in this course.



Q: Is DAICO free, and where is it located?
  routed to [faq] (question #2)
  DAICO is free, as course seats are sponsored by SDAIA, so there is no tuition fee for accepted learners. It is located at King Saud University (KSU).



Q: What does the capstone need in order to pass?
  routed to [capstone_mentor] (question #3)
  The capstone needs a minimum score of 60 out of 100 to pass, with no single rubric section scoring below 40% of its points.

Cross-thread memory: Yousef's question count across 3 different threads = 3


## Write-up - one paragraph per rubric section

**§1 Agent fundamentals.** Four tools (`search_courses`, `get_course_details`, `check_seat_availability`,
`get_prerequisites`) read the real `courses.json` and act on their arguments; the model chooses them in a
tool-calling loop. `CourseRecommendation` (via `with_structured_output`) returns a Pydantic object that code
parses.

**§2 Multi-agent / routing (Track A).** A supervisor LLM classifies each message with `with_structured_output`
into a constrained `Route` (four workers). The decision is the model's - there is no keyword matching anywhere.

**§3 RAG pipeline (Hybrid).** DAICO docs are loaded, split with `RecursiveCharacterTextSplitter`, embedded with
`text-embedding-3-small`, stored in `InMemoryVectorStore`, and retrieved. A verbatim-answer probe (the 80%
attendance rule) confirms retrieval works. We justify **Hybrid RAG**: structured facts go to tools, prose to
vector search, and the supervisor picks per message.

**§4 Context & state.** `InMemorySaver` (thread_id) holds short-term conversation state; a separate
`InMemoryStore` holds long-term learner facts. The cross-thread test writes an interest in `conv-A` and reads
it back in `conv-B`, with assertions proving persistence - not a growing message list.

**§5 Human-in-the-loop.** `enroll_task` calls `interrupt()` before reserving a sponsored seat; the notebook
shows the paused payload and then `Command(resume="approve")` completing the enrollment. Both halves run.

**§6 Functional API & error handling.** Every workflow uses `@task` / `@entrypoint`. Two error strategies are
implemented: a real `RetryPolicy` on `fetch_seat_snapshot` (transient retry, recovers on attempt 3) and a
`ValidationError` feedback loop in `classify_message` (LLM-recoverable).

**§7 Workflow pattern.** We use the **Evaluator-Optimizer** pattern for the Capstone Mentor: draft feedback,
judge it against the rubric with structured output, and revise if the verdict is `revise`.

**§8 LangSmith observability.** Tracing is wired to the correct `LANGCHAIN_TRACING_V2` variable and was run with
a real LangSmith key, sending traces to the `daico-navigator` project. The trace showed each `daico_navigator`
run decomposing into its steps - `bump_and_get_count`, then the supervisor `classify_message` call, then the
selected worker. The supervisor classification is a small, fast call (~170-380 prompt / ~20-40 completion tokens,
under ~1s), while the **Course Advisor path is the clear hotspot**: its tool-calling loop issues several
ChatOpenAI calls, pushing a single root run to ~1k tokens and several seconds, versus FAQ/Capstone answers which
cost only a classify + one grounded call (~150 tokens, ~1s). The trace made the cost of the multi-step tool loop
obvious at a glance.